# Raw LLM Client + LiteLLM Comparison

**Concept primer.** LLM APIs are `stateless`: every request is independent, and the model has no
memory of previous calls. What *feels* like a conversation is really you resending the entire
message history (system, user, and assistant turns) on every single request. The model reads that
list fresh each time and generates the next turn. Once you understand this, provider-swapping
becomes easy: as long as you keep sending the same message-list shape, you can point the same logic
at a different provider or model.

**LiteLLM** gives you one `completion()` function that works the
same way across Groq, OpenAI, Together, Ollama, and others — you only change the `model=` string.

**Learning objectives**
- Explain why LLM APIs are stateless and how conversation history simulates memory
- Call an LLM directly, then via LiteLLM to abstract across providers
- Compare cost, latency, and token usage across providers for the same prompt


## 1. Guided setup
Run this cell as-is — it installs dependencies, loads your `.env` (if it's uploaded along with this Colab notebook), and checks which keys are
available so the lab doesn't start on a broken environment.

> Note: Groq is the required provider here since its free tier needs no billing setup. OpenAI is optional — Part 3's comparison will include it automatically if `OPENAI_API_KEY` is set.


## Get a Groq API key (free, no credit card)

1. Go to [console.groq.com](https://console.groq.com) and sign up (email, Google, or GitHub).
2. Click the hamburger menu → **API Keys** (or go directly to [console.groq.com/keys](https://console.groq.com/keys)).
3. Click **Create API Key**, give it a name, and copy the key immediately — it starts with `gsk_` and is shown only once.
4. Run the cell below once to generate a `.env` file, paste your key into it, save, then re-run the cell to confirm it's picked up.

> Note: For more detailed content and guide options, please refer to the official [Groq Documentation](https://console.groq.com/docs/overview).

In [1]:
# Guided setup (pre-written — do not edit except the line below)
%pip install -q --break-system-packages "litellm==1.93.0" "python-dotenv==1.2.2" "tiktoken==0.13.0" "groq==0.34.0"

import os

GROQ_API_KEY = "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"   # <-- paste your gsk_... key here, between the quotes
OPENAI_API_KEY = ""  # <-- optional

if not GROQ_API_KEY:
    print("GROQ_API_KEY is empty — paste your key above, between the quotes, then re-run this cell.")
else:
    os.environ["GROQ_API_KEY"] = GROQ_API_KEY
    print("Environment OK — GROQ_API_KEY found.")

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print(f"OPENAI_API_KEY: {'found' if os.getenv('OPENAI_API_KEY') else 'not set (Part 3 will use Groq only)'}")

"""
EXPECTED OUTPUT
---------------
GROQ_API_KEY is empty — paste your key above, between the quotes, then re-run this cell.
OPENAI_API_KEY: not set (Part 3 will use Groq only)
--- (after pasting a real key and re-running) ---
Environment OK — GROQ_API_KEY found.
"""

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 946.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 12.7 MB/s eta 0:00:00
Environment OK — GROQ_API_KEY found.
OPENAI_API_KEY: not set (Part 3 will use Groq only)


'\nEXPECTED OUTPUT\n---------------\nGROQ_API_KEY is empty — paste your key above, between the quotes, then re-run this cell.\nOPENAI_API_KEY: not set (Part 3 will use Groq only)\n--- (after pasting a real key and re-running) ---\nEnvironment OK — GROQ_API_KEY found.\n'

## 2. Basic LLM client

**Concept:** Every LLM provider exposes its own Software Development Kit (SDK) or HTTP API. A request typically includes a model name, a list of messages, and generation parameters such as temperature. At this stage, each provider has its own syntax and client implementation.

**Question:** Your task is to send a simple prompt to an LLM using the provider SDK and display the generated response.


In [2]:
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

# Call the LLM once and print the reply
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "In one short sentence, say hello and name yourself."}],
)
print(response.choices[0].message.content)

"""
EXPECTED OUTPUT (shape, not exact wording — live model call)
---------------
A single short sentence of text.
"""

Hello, my name is Ada, and I'm here to help.


'\nEXPECTED OUTPUT (shape, not exact wording — live model call)\n---------------\nA single short sentence of text.\n'

In [8]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# Call the LLM once and print the reply
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "In one short sentence, say hello and name yourself."}],
)
print(response.choices[0].message.content)

"""
EXPECTED OUTPUT (shape, not exact wording — this is a live model call)
---------------
A single short sentence of text, e.g.:
Hello! I'm an AI assistant here to help.
"""


Hello, I am an AI assistant, and I'm here to help with any questions you may have.


"\nEXPECTED OUTPUT (shape, not exact wording — this is a live model call)\n---------------\nA single short sentence of text, e.g.:\nHello! I'm an AI assistant here to help.\n"

In [5]:
# === Self-check ===
assert response is not None, "Expected a response object"
assert len(response.choices[0].message.content) > 0, "Reply should not be empty"
print("basic client: OK")

"""
EXPECTED OUTPUT
---------------
basic client: OK
"""


basic client: OK


'\nEXPECTED OUTPUT\n---------------\nbasic client: OK\n'

## 3. Adding conversation history

**Concept:** LLM APIs are stateless — the model has no memory between calls. Every request must
include the full conversation history as a list of messages, each shaped like
`{"role": "user"/"assistant"/"system", "content": "..."}`. The model reads that entire list fresh
on every call and generates the next turn from it. Whatever "memory" the conversation appears to
have is really just you resending the growing `messages` list on every call.

**Question:** Let's extend the client into a function that appends the user's message to a running
history, calls the LLM with that full history, appends the model's reply back to the history, and
returns the reply — so a later call can still "remember" what was said earlier.

**Your task:** implement `bare_chat(user_input, messages)` so that it:
1. Appends a new `{"role": "user", "content": user_input}` dict to `messages`.
2. Calls `client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)`.
3. Pulls the reply text out of the response (`response.choices[0].message.content`).
4. Appends `{"role": "assistant", "content": reply}` to `messages`.
5. Returns the reply string.


In [6]:
messages = [
    {"role": "system", "content": "You are a concise, helpful assistant."}
]

def bare_chat(user_input: str, messages: list) -> str:
    """Append the user turn, call the API with the full history, append the assistant reply, return it."""
    # Implement bare_chat
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

name = "Alex"
print(bare_chat(f"My name is {name}. Remember that.", messages))
print(bare_chat("What is my name?", messages))

"""
EXPECTED OUTPUT (shape, not exact wording)
---------------
Some acknowledgement of the name, e.g. "Got it, I'll remember that, Alex!"
A reply that correctly recalls "Alex", e.g. "Your name is Alex."
"""


Nice to meet you, Alex. I'll remember your name for our conversation. How can I assist you today?
Your name is Alex.


'\nEXPECTED OUTPUT (shape, not exact wording)\n---------------\nSome acknowledgement of the name, e.g. "Got it, I\'ll remember that, Alex!"\nA reply that correctly recalls "Alex", e.g. "Your name is Alex."\n'

In [7]:
# === Self-check ===
# The model should reference "Alex" in its second reply, proving the history (not model memory)
# is what carries context across calls.
assert len(messages) == 5, f"Expected 5 messages in history (system + 2 user + 2 assistant), got {len(messages)}"
assert "alex" in messages[-1]["content"].lower(), "Second reply should recall the name from history"
print("bare_chat: OK")

"""
EXPECTED OUTPUT
---------------
bare_chat: OK
"""


bare_chat: OK


'\nEXPECTED OUTPUT\n---------------\nbare_chat: OK\n'

## 4. Session state — one history per user, not one global list

**Concept:** `bare_chat` used a single `messages` list — fine for one user, but a real service talks to many users at once, each with their own conversation. A **session store** is just a dictionary mapping a `session_id` to that user's own `messages` list, so conversations don't
bleed into each other. This is also exactly what Day 4's `session_id` tagging built on top of — same idea, now generalized to hold more than one at a time.

**Task:** implement `SessionStore` with `get_or_create(session_id)` (returns that session's message list, creating a fresh one with a system prompt if it doesn't exist yet) and `chat(session_id, user_input)` (does what `bare_chat` did, but scoped to one session).

In [8]:
class SessionStore:
    def __init__(self):
        self._sessions: dict[str, list] = {}

    def get_or_create(self, session_id: str) -> list:
        if session_id not in self._sessions:
            self._sessions[session_id] = [{"role": "system", "content": "You are a concise, helpful assistant."}]
        return self._sessions[session_id]

    def chat(self, session_id: str, user_input: str) -> str:
        history = self.get_or_create(session_id)
        history.append({"role": "user", "content": user_input})
        response = client.chat.completions.create(model="llama-3.1-8b-instant", messages=history)
        reply = response.choices[0].message.content
        history.append({"role": "assistant", "content": reply})
        return reply

store = SessionStore()
print(store.chat("user-alex", "My name is Alex."))
print(store.chat("user-priya", "My name is Priya."))
print(store.chat("user-alex", "What is my name?"))     # should say Alex, not Priya

"""
EXPECTED OUTPUT (shape, not exact wording)
---------------
Some acknowledgement for Alex
Some acknowledgement for Priya
A reply that says "Alex" — proving user-alex's history never mixed with user-priya's
"""

Hi Alex. What's on your mind, or how can I assist you today?
Nice to meet you, Priya! How can I assist you today?
Your name is Alex.


'\nEXPECTED OUTPUT (shape, not exact wording)\n---------------\nSome acknowledgement for Alex\nSome acknowledgement for Priya\nA reply that says "Alex" — proving user-alex\'s history never mixed with user-priya\'s\n'

In [9]:
# Self-check
assert "alex" in store.get_or_create("user-alex")[-1]["content"].lower()
assert len(store._sessions) == 2
print("SessionStore: OK — two independent histories, no cross-talk.")

SessionStore: OK — two independent histories, no cross-talk.


## Understanding the `completion()` call

A single `completion()` call is really a contract with several independent knobs. This section
walks four of them: what you say (Instructions & Input), how randomly the model answers
(Generation hyperparameters), whether/how it "thinks" before answering (Reasoning), and what
shape the reply comes back in (Output Format).
Reference: https://docs.litellm.ai/docs/completion/input

### 4a. Instructions & Input

**Concept:** Every message has a `role`.
- `system` sets standing behavior for the whole conversation (tone, constraints, persona) and is read once, before anything else.
- `user` is the actual ask.
- `assistant` is the model's own prior replies.

The `system` message is the cheapest lever you have: change it, and the same question can come back formatted totally differently, in a different tone, or restricted to a different length — without touching your code's logic.

For more information regarding the input parameters, refer to [LiteLLM Documentation - Input Params](https://docs.litellm.ai/docs/completion/input)

**Task:** Build a `messages` list with a `system` role that constrains the reply's length/format, and a `user` role with the actual question. Run it, then try changing just the system message and re-running to see how much the reply's shape changes for the same question.

In [3]:
from litellm import completion

# Build a messages list with a system role (sets behavior) and a user role (the actual ask).
messages = [
    {"role": "system", "content": "You always answer in exactly one sentence, no more."},
    {"role": "user", "content": "What does a circuit breaker do?"},
]
resp = completion(model="groq/llama-3.1-8b-instant", messages=messages)
print(resp.choices[0].message.content)

"""
EXPECTED OUTPUT (shape)
---------------
A single sentence answering the question.
"""

A circuit breaker protects an electrical circuit from damage by automatically turning off the power supply when it detects an overcurrent or short circuit.


'\nEXPECTED OUTPUT (shape)\n---------------\nA single sentence answering the question.\n'

In [11]:
# Self-check
assert resp.choices[0].message.content.count(".") <= 2, "system instruction should keep it to ~1 sentence"
print("Instructions & Input: OK")

Instructions & Input: OK


### 4b. Generation hyperparameters

**Concept:** These knobs don't change *what* the model knows — they change *how* it samples the next word.
- `temperature` (0–2) controls randomness: near 0 is nearly deterministic; higher values are more varied at the cost of consistency.
- `max_tokens` caps reply length (hitting the cap shows
- `finish_reason="length"` instead of `"stop"`)
- `stop` ends generation immediately on a matching
string
- `seed` asks for (not-guaranteed) reproducible randomness.

For more information regarding the input parameters, refer to [LiteLLM Documentation - Input Params](https://docs.litellm.ai/docs/completion/input)

**Task:** Call the same prompt twice with `temperature=0`, a `max_tokens` cap, and a `stop` sequence, and confirm the two replies are effectively identical. Then check `finish_reason`.

In [12]:
# Call the same prompt twice with temperature=0 (deterministic-ish) and observe stop/max_tokens too.
prompt = "List exactly 3 colors."
r1 = completion(model="groq/llama-3.1-8b-instant", messages=[{"role": "user", "content": prompt}],
                 temperature=0, max_tokens=30, stop=["\n\n"])
r2 = completion(model="groq/llama-3.1-8b-instant", messages=[{"role": "user", "content": prompt}],
                 temperature=0, max_tokens=30, stop=["\n\n"])
print(r1.choices[0].message.content)
print(r2.choices[0].message.content)

"""
EXPECTED OUTPUT (shape)
---------------
Two replies that are identical or near-identical (temperature=0 minimizes randomness).
"""

1. Blue
2. Red
3. Green
1. Blue
2. Red
3. Green


'\nEXPECTED OUTPUT (shape)\n---------------\nTwo replies that are identical or near-identical (temperature=0 minimizes randomness).\n'

### Additional generation parameters

Besides `temperature` and `max_tokens`, most OpenAI-compatible APIs support several useful options.

- **stream**
  - Returns tokens as they are generated instead of waiting for the complete response.
  - Useful for chat interfaces to reduce perceived latency.

- **seed**
  - Requests deterministic generation when the provider supports it.
  - Helpful for debugging and reproducible experiments.

- **logprobs**
  - Returns the probability assigned to generated tokens.
  - Useful for confidence estimation, debugging, and evaluation.

- **top_p**
  - Alternative sampling strategy called nucleus sampling.
  - Usually adjust either `temperature` or `top_p`, not both.

- **frequency_penalty**
  - Reduces repeated words or phrases.

- **presence_penalty**
  - Encourages introducing new topics instead of repeating previous ones.

In [4]:
from litellm import completion

resp = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[
        {"role": "user", "content": "Write one interesting fact about space."}
    ],
    temperature=0.2,
    max_tokens=60,
    seed=42,
)

print(resp.choices[0].message.content)

One interesting fact about space is that there is a giant storm on Jupiter that has been raging for at least 187 years. The Great Red Spot, as it's known, is a persistent anticyclonic storm on Jupiter, which means it's a high-pressure region with clockwise rotation. It's


### 4c. Reasoning

**Concept:** Some models generate an internal chain-of-thought before their final answer — useful
for multi-step problems where "thinking out loud" first measurably improves accuracy. The
reasoning trace and the final answer come back as two separate fields (field name varies by
provider — check `resp.choices[0].message.model_dump()` if unsure rather than assuming from docs).

Reference: [LiteLLM Documentation - Thinking / Reasoning Content](https://docs.litellm.ai/docs/reasoning_content)

**Task (demo, read and run):** call a Groq-hosted reasoning model on a multi-step word problem and print the reasoning trace and the final answer side by side — notice the answer is usually much shorter than the reasoning that produced it.

In [5]:
resp = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "A train leaves at 2pm going 60mph. Another leaves the same station at 3pm going 90mph, same direction. When does the second train catch up?"}],
    reasoning_effort="low",
)
print("--- reasoning (the model's scratch work) ---")
print(resp.choices[0].message.reasoning)
print("\n--- content (the final answer) ---")
print(resp.choices[0].message.content)

--- reasoning (the model's scratch work) ---
We solve: first train leaves at 2pm, speed 60 mph. By 3pm, it has traveled 60 miles. Second leaves at 3pm, speed 90 mph, relative speed 30 mph. Distance gap 60 miles. Time to close = 60/30 = 2 hours. So catches at 5pm.

--- content (the final answer) ---
The first train gets a one‑hour head start.

* **First train:**  
  * Leaves at 2 pm  
  * Speed = 60 mph  
  * Distance traveled by 3 pm = 60 mi × 1 h = **60 mi**

* **Second train:**  
  * Leaves at 3 pm  
  * Speed = 90 mph  

From 3 pm onward the second train is closing the gap at the **relative speed**:

\[
\text{Relative speed}=90\text{ mph} - 60\text{ mph}=30\text{ mph}
\]

The gap to close is 60 mi, so the time required is:

\[
\text{Time} = \frac{\text{distance}}{\text{relative speed}} = \frac{60\text{ mi}}{30\text{ mph}} = 2\text{ h}
\]

Adding those 2 hours to the departure time of the second train (3 pm) gives:

\[
3\text{ pm} + 2\text{ h} = \boxed{5\text{ pm}}
\]

Thus, the seco

In [6]:
json_mode = completion(model="groq/llama-3.1-8b-instant",
                        messages=[{"role": "user", "content": "Return a JSON object like {\"color\": \"...\"} for one color."}],
                        response_format={"type": "json_object"})
print("json mode:", json_mode.choices[0].message.content)

json mode: {
  "color": "Blue"
}


### 4d. Output Format

**Concept:** By default a model returns free text. `{"type": "json_object"}` ("JSON mode") tells it to return *some* valid JSON, but doesn't enforce which fields — you still validate the shape yourself. `{"type": "json_schema", ..., "strict": true}` goes further: the model is constrained at the token level so its output is guaranteed to match your schema exactly.

**Gotcha:** JSON mode requires the literal word "json" somewhere in your messages — Groq (like
OpenAI) rejects the request with a 400 error otherwise, even if braces/quotes clearly imply JSON.

Reference: [LiteLLM Documentation - Structured Outputs (JSON Mode)](https://docs.litellm.ai/docs/completion/json_mode)

**Task (demo, read and run):** compare plain text, JSON mode, and strict JSON Schema for a similar request, and notice how each gives you progressively less to validate yourself.

In [9]:
plain = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Name one color."}]
)
print("plain text:", plain.choices[0].message.content)

json_mode = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Return a JSON object like {\"color\": \"...\"} for one color."}],
    response_format={"type": "json_object"}
)
print("json mode:", json_mode.choices[0].message.content)

schema_resp = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Name one color and how confident you are (0-1)."}],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "color_pick",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {"color": {"type": "string"}, "confidence": {"type": "number"}},
                "required": ["color", "confidence"],
                "additionalProperties": False,
            },
        },
    },
)
print("strict schema:", schema_resp.choices[0].message.content)

plain text: Blue.
json mode: {
  "color": "Blue"
}
strict schema: {"color":"blue","confidence":0.9}


##4e. Prompt templating

Rather than manually building prompts with string concatenation, applications typically use
prompt templates. A template contains placeholders that are filled with user input at runtime.

Benefits:
- Cleaner prompts
- Reusable instructions
- Easier maintenance
- Reduced prompt duplication

In [4]:
from string import Template

template = Template("""
Answer the following question in one sentence.

Question:
$question
""")

prompt = template.substitute(
    question="Why is the sky blue?"
)

response = completion(
    model="groq/llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}],
)

print(response.choices[0].message.content)

The sky appears blue due to a phenomenon called scattering, where shorter (blue) wavelengths of light are scattered more by the tiny molecules of gases in the atmosphere than longer (red) wavelengths.


## 5. Refactor to LiteLLM across providers

**Concept:** Working with multiple provider SDKs increases development effort because each provider has its own API syntax. LiteLLM provides a common interface that abstracts provider-specific implementations. Switching providers typically requires changing only the model identifier.

LiteLLM provides one function, `completion()`, that works across every major LLM provider using the same `messages` format. The only thing that changes between providers is the `model=` string (e.g. `"groq/llama-3.1-8b-instant"`, `"gpt-4o-mini"`). This lets the same calling code run against any provider without rewriting request logic.

**Question:** Let's write a client function that calls the LLM through LiteLLM using whichever `model` string is passed in, and returns just the reply text.

**Your task:** implement `litellm_chat(model, user_input)` so that it:
1. Calls `completion(model=model, messages=[...])` with a system message ("You are a concise,
   helpful assistant.") and a user message containing `user_input`.
2. Returns the reply text from the response (`resp.choices[0].message.content`).


In [5]:
from litellm import completion

def litellm_chat(model: str, user_input: str) -> str:
    # Implement litellm_chat
    resp = completion(
        model=model,
        messages=[
            {"role": "system", "content": "You are a concise, helpful assistant."},
            {"role": "user", "content": user_input},
        ],
    )
    return resp.choices[0].message.content

models_to_try = ["groq/llama-3.1-8b-instant"]
if os.getenv("OPENAI_API_KEY"):
    models_to_try.append("gpt-4o-mini")

prompt = "In one sentence, explain why LLM APIs are stateless."
results = {}
for m in models_to_try:
    results[m] = litellm_chat(m, prompt)
    print(f"[{m}] {results[m]}")

"""
EXPECTED OUTPUT (shape, not exact wording)
---------------
[groq/llama-3.1-8b-instant] <one-sentence reply>
[gpt-4o-mini] <one-sentence reply>   (only if OPENAI_API_KEY is set)
"""


[groq/llama-3.1-8b-instant] LLM (Large Language Model) APIs are designed to be stateless because they process requests independently, caching model outputs and managing state locally is difficult, and ensuring model updates and consistency across multiple requests is challenging in a stateless architecture.


'\nEXPECTED OUTPUT (shape, not exact wording)\n---------------\n[groq/llama-3.1-8b-instant] <one-sentence reply>\n[gpt-4o-mini] <one-sentence reply>   (only if OPENAI_API_KEY is set)\n'

In [6]:
# === Self-check ===
assert len(results) >= 1, "Expected at least one provider result"
for m, r in results.items():
    assert isinstance(r, str) and len(r) > 0, f"Empty response from {m}"
print("litellm_chat: OK ->", list(results))

"""
EXPECTED OUTPUT
---------------
litellm_chat: OK -> ['groq/llama-3.1-8b-instant']   (plus 'gpt-4o-mini' if OPENAI_API_KEY is set)
"""


litellm_chat: OK -> ['groq/llama-3.1-8b-instant']


"\nEXPECTED OUTPUT\n---------------\nlitellm_chat: OK -> ['groq/llama-3.1-8b-instant']   (plus 'gpt-4o-mini' if OPENAI_API_KEY is set)\n"

In [7]:
def render_template(template: list[dict], **variables) -> list[dict]:
    return [{"role": m["role"], "content": m["content"].format(**variables)} for m in template]

RISK_SUMMARY_TEMPLATE = [
    {"role": "system", "content": "You always answer in exactly one sentence, no more."},
    {"role": "user", "content": "What does a {concept} do?"},
]

messages = render_template(RISK_SUMMARY_TEMPLATE, concept="circuit breaker")
resp = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages)
print(resp.choices[0].message.content)

# Reuse the SAME template with a different variable — no rewriting the messages list
messages2 = render_template(RISK_SUMMARY_TEMPLATE, concept="load balancer")
resp2 = client.chat.completions.create(model="llama-3.1-8b-instant", messages=messages2)
print(resp2.choices[0].message.content)

A circuit breaker is an electrical device that automatically trips and interrupts the flow of electrical current when it senses excessive current draw or a fault, thereby preventing overheating, electrical fires, and other potential hazards.
A load balancer distributes incoming network traffic across multiple servers to improve responsiveness, reliability, and scalability by preventing any one server from becoming overwhelmed with requests.


## 6. Part 3 — Cost, latency, and token comparison

**Concept:** Comparing providers requires three measurements for the same prompt: how long the
call takes (latency), how many tokens went in (input), and how many came out (output). LiteLLM's
`token_counter()` computes token counts for a given model without an extra API call.

**Question:** Let's time each provider's call and count its input/output tokens, then collect the
results into `comparison` so we can print a side-by-side table.

**Your task:** for each model in `models_to_try`, inside the loop:
1. Record `start = time.time()` before calling `completion()`, and compute `latency = time.time() - start` after.
2. Count input tokens: `token_counter(model=m, messages=msgs)`.
3. Count output tokens: `token_counter(model=m, text=<the reply text>)`.
4. Append a dict `{"model": m, "latency_sec": ..., "input_tokens": ..., "output_tokens": ...}` to `comparison`.


In [8]:
from litellm import token_counter
import time

comparison = []
for m in models_to_try:
    msgs = [{"role": "user", "content": prompt}]
    # Time the call, count tokens, append a row to `comparison` ===
    start = time.time()
    resp = completion(model=m, messages=msgs)
    latency = time.time() - start
    reply_text = resp.choices[0].message.content
    input_tokens = token_counter(model=m, messages=msgs)
    output_tokens = token_counter(model=m, text=reply_text)
    comparison.append({
        "model": m,
        "latency_sec": latency,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    })

for row in comparison:
    print(row)

"""
EXPECTED OUTPUT (shape, not exact numbers — real latency/token counts)
---------------
{'model': 'groq/llama-3.1-8b-instant', 'latency_sec': <float>, 'input_tokens': <int>, 'output_tokens': <int>}
"""


tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

{'model': 'groq/llama-3.1-8b-instant', 'latency_sec': 0.30259108543395996, 'input_tokens': 20, 'output_tokens': 47}


"\nEXPECTED OUTPUT (shape, not exact numbers — real latency/token counts)\n---------------\n{'model': 'groq/llama-3.1-8b-instant', 'latency_sec': <float>, 'input_tokens': <int>, 'output_tokens': <int>}\n"

In [ ]:
# === Self-check ===
assert len(comparison) == len(models_to_try), "Expected one comparison row per model"
required_fields = {"model", "latency_sec", "input_tokens", "output_tokens"}
for row in comparison:
    assert required_fields.issubset(row.keys()), f"Missing fields in {row}"
print("comparison: OK")

"""
EXPECTED OUTPUT
---------------
comparison: OK
"""


# Structured Output Parser (Invoice -> Pydantic)

**Concept primer.** LLMs generate free-form text by default, which is unreliable for downstream
code that expects a fixed shape. Structured outputs solve this: you define the exact shape you
want as a **Pydantic model**, ask the LLM to return JSON matching that shape (JSON mode), then
**validate** the response with Pydantic. Real-world text is messy — typos, missing fields,
ambiguous currencies — so validation will sometimes fail. A **repair loop** feeds the validation
error back to the model and asks it to try again, which is far more reliable than a single
unchecked pass.

**Learning objective:** design a Pydantic schema and coerce free-form model output into validated
objects, with a repair loop for malformed output.

> **Note:** The 18 sample records here were written to be genuinely messy (typos, missing totals, two currencies in one line, deliberately wrong math in #10) specifically so some of them **should** fail validation and exercise the repair path — a 18/18 success rate is not the goal; understanding *which* records fail and *why* is.


## 1. Guided setup
Run this cell as-is — it installs dependencies, loads your `.env`, and loads the sample invoice
data so the lab doesn't start on a broken environment.

*(Groq is the required provider — its free tier needs no billing setup.)*


In [14]:
# === Guided setup (pre-written — do not edit) ===
%pip install -q --break-system-packages "litellm==1.93.0" "pydantic==2.13.4" "python-dotenv==1.2.2"

import os, json, time
from pathlib import Path
from dotenv import load_dotenv
from pydantic import BaseModel, ValidationError, Field
from litellm import completion

env_path = Path.cwd() / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
else:
    print(f"No .env file found at {env_path}. Copy .env.template to .env and add your API key.")
    print("If you already set the key in your shell, this cell will still proceed.")

required_keys = ["GROQ_API_KEY"]
missing = [k for k in required_keys if not os.getenv(k)]
if missing:
    print(f"Missing required keys: {missing}. Add them to your .env file before continuing.")
else:
    print("Environment OK — required keys found.")

with open("data/sample_invoices.json") as f:
    invoices = json.load(f)
print(f"Loaded {len(invoices)} sample invoice/email records.")
print(invoices[0])

"""
EXPECTED OUTPUT
---------------
Environment OK — required keys found.
Loaded 18 sample invoice/email records.
{'id': 1, 'text': "Hey, plz find invoic attached. Invoice # INV-1001. Bill to: Acme Corp. ..."}
"""


No .env file found at /content/.env. Copy .env.template to .env and add your API key.
If you already set the key in your shell, this cell will still proceed.
Environment OK — required keys found.
Loaded 18 sample invoice/email records.
{'id': 1, 'text': 'Hey, plz find invoic attached. Invoice # INV-1001. Bill to: Acme Corp. Item: Laptop Stand x3 @ $25 ea. Item: USB-C Hub x2 @ $18. Total due: $86. Thx - Raj'}


'\nEXPECTED OUTPUT\n---------------\nEnvironment OK — required keys found.\nLoaded 18 sample invoice/email records.\n{\'id\': 1, \'text\': "Hey, plz find invoic attached. Invoice # INV-1001. Bill to: Acme Corp. ..."}\n'

In [10]:
import os, json
os.makedirs("data", exist_ok=True)
invoices_data = [
    {"id": 1, "text": "Hey, plz find invoic attached. Invoice # INV-1001. Bill to: Acme Corp. Item: Laptop Stand x3 @ $25 ea. Item: USB-C Hub x2 @ $18. Total due: $86. Thx - Raj"},
    {"id": 2, "text": "INVOICE\nInvoice No: 1002\nCustomer: Bluewave Logistics\nDate: 03/14/2026\n1. Office Chair - Qty 5 - Unit Price 120 USD\n2. Desk Lamp - Qty 10 - Unit Price 15 USD\nGrand Total: 750 USD"},
    {"id": 3, "text": "hi team, sending the bill. inv#1003. client: Nimbus Retail Pvt Ltd. products: Wireless Mouse (qty 4, price 12), Keyboard (qty 4, price 22). amount payable = 136"},
    {"id": 4, "text": "Invoice ID: 1004\nBilled To: Sunrise Traders\nLine items:\n- Monitor 24in x2 @ 8500 INR\n- HDMI Cable x2 @ 300 INR\nTotal: 17600 INR"},
    {"id": 5, "text": "quick invoice for u - #1005 - Global Foods Inc - stuff bought: Packaging Boxes qty 100 rate 2 each, Tape Rolls qty 20 rate 3 each - total = 260 (currency not mentioned, assume same as last time??)"},
    {"id": 6, "text": "INVOICE #1006\nBill To: Northstar Manufacturing\nItems:\n1) Steel Bolts - 500 units - $0.10/unit\n2) Washers - 500 units - $0.05/unit\nSubtotal: $75\nTax (8%): $6\nTotal: $81"},
    {"id": 7, "text": "yo here's the invoice lol. no inv number given oops. client = Delta Print Shop. items: A4 Paper Ream x10 @ 5, Toner Cartridge x1 @ 45. total 95"},
    {"id": 8, "text": "Invoice: 1008\nCustomer: EuroTech GmbH\nItem 1: Router x3, price 45 EUR each\nItem 2: Ethernet Cable x6, price 3 EUR each\nItem 3: Switch x1, price 60 EUR\nTotal amount: 213 EUR\nNote: partial payment of 100 EUR already received, remaining not stated clearly"},
    {"id": 9, "text": "billing doc - inv no 1009 - customer Pacific Rim Traders - product: Bamboo Cutting Board qty 8 price $9 - product: Ceramic Bowl qty 12 price $6 (missing total field, pls compute)"},
    {"id": 10, "text": "INVOICE #1010\nBill To: Redwood Analytics\nLine Items:\n- Consulting Hours: 10 hrs @ $150/hr\n- Software License: 1 @ $500\nTotal Due: $2000\n(math looks off on purpose - repair loop should catch this)"},
    {"id": 11, "text": "inv 1011, client: Maple Leaf Hardware, items: Hammer x5 @ 15 CAD, Screwdriver Set x3 @ 22 CAD, Total: 141 CAD, also 50 USD shipping fee added separately - two currencies mixed"},
    {"id": 12, "text": "Invoice Number: 1012\nBilled to: Sahara Textiles\nItems purchased:\n- Cotton Fabric (meters): 200 @ 3.5\n- Thread Spools: 50 @ 1.2\nTotal: 760\ncurrency: AED"},
    {"id": 13, "text": "hey quick one - inv#1013 - Lotus Bakery Supplies - flour bags qty 30 price 4 each - sugar bags qty 20 price 3 each - (no total given at all, no tax mentioned)"},
    {"id": 14, "text": "INVOICE #1014\nCustomer: Falcon Aerospace Parts\nItems:\n1. Titanium Bracket x2 @ $340\n2. Rivets (box of 100) x5 @ $28\nSubtotal: $820\nTax: $65.60\nTotal: $885.60"},
    {"id": 15, "text": "billng info, inv num 1015, client Cedar Grove Furniture, item: Oak Table x1 price 450, item: Oak Chair x6 price 80, curency: mixed USD and CAD mentioned in same line - ambiguous, total not computed"},
    {"id": 16, "text": "Invoice: 1016\nBill To: Orion Med Supplies\n(no line items listed, only a lump sum) Total charge for misc medical consumables: $430"},
    {"id": 17, "text": "inv# 1017 - client: Harbor View Hotel - Item: Bath Towels qty 40 @ 6 - Item: Bedsheets qty 20 @ 15 - total: 540 - typo: 'toal' instead of total appears once in original doc"},
    {"id": 18, "text": "INVOICE NO 1018\nCUSTOMER: Vertex Chemicals Ltd\nITEMS:\n- Solvent Drum x2 @ 220 USD\n- Safety Gloves (box) x10 @ 8 USD\nTOTAL: 520 USD"},
]
with open("data/sample_invoices.json", "w") as f:
    json.dump(invoices_data, f)
print("Created data/sample_invoices.json")

Created data/sample_invoices.json


## 2. Define the Pydantic schema

**Concept:** A Pydantic model defines the exact field names and types expected in a piece of data.
When you construct `Invoice(**data)`, Pydantic checks `data` against these field definitions and
raises `ValidationError` on any mismatch (wrong type, missing required field). Marking a field as
`| None = None` makes it optional, so records missing that value don't fail validation for that
reason alone.

**Question:** Let's define what an invoice line item and a full invoice look like, so we have
something concrete to validate the model's output against.

**Task:**
1. In `LineItem`, define three fields: `description: str`, `quantity: float`, `unit_price: float`.
2. In `Invoice`, define five fields: `invoice_number: str | None = None`, `customer: str`, `currency: str | None = None`, `line_items: list[LineItem]`, `total: float | None = None`.


In [11]:
class LineItem(BaseModel):
    description: str
    quantity: float
    unit_price: float

class Invoice(BaseModel):
    invoice_number: str | None = None
    customer: str
    currency: str | None = None
    line_items: list[LineItem]
    total: float | None = None

In [12]:
# === Self-check (new — was missing before) ===
li_fields = set(LineItem.model_fields)
inv_fields = set(Invoice.model_fields)
assert li_fields == {"description", "quantity", "unit_price"}, f"LineItem fields: {li_fields}"
assert inv_fields == {"invoice_number", "customer", "currency", "line_items", "total"}, f"Invoice fields: {inv_fields}"
assert Invoice.model_fields["customer"].is_required(), "customer must be required"
assert not Invoice.model_fields["invoice_number"].is_required(), "invoice_number must be optional"
assert not Invoice.model_fields["total"].is_required(), "total must be optional"
print("schema: OK ->", sorted(inv_fields))

"""
EXPECTED OUTPUT
---------------
schema: OK -> ['currency', 'customer', 'invoice_number', 'line_items', 'total']
"""


schema: OK -> ['currency', 'customer', 'invoice_number', 'line_items', 'total']


"\nEXPECTED OUTPUT\n---------------\nschema: OK -> ['currency', 'customer', 'invoice_number', 'line_items', 'total']\n"

## 3. Extract with the LLM

**Concept:** `response_format={"type": "json_object"}` instructs the model to return valid JSON
rather than free-form text ("JSON mode"). It guarantees parseable JSON but not that the JSON
matches any particular schema — the field names still come entirely from the prompt.

**Question:** Let's write a function that sends the messy invoice text to the LLM and gets back
its best attempt at structured JSON.

**Task:** Implement `extract_raw(text)`:
1. Call `completion(model="groq/llama-3.1-8b-instant", messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(text=text)}], response_format={"type": "json_object"})`.
2. Return `resp.choices[0].message.content`.


In [15]:
EXTRACTION_PROMPT = """Extract invoice data from the text below into JSON with exactly these fields:
invoice_number (string or null), customer (string), currency (string or null),
line_items (list of objects with description, quantity, unit_price), total (number or null).
Return ONLY valid JSON, no markdown fences, no commentary.

Text:
{text}"""

def extract_raw(text: str) -> str:
    # Implement extraction from raw data
    resp = completion(
        model="groq/llama-3.1-8b-instant",
        messages=[{"role": "user", "content": EXTRACTION_PROMPT.format(text=text)}],
        response_format={"type": "json_object"},
    )
    return resp.choices[0].message.content

raw = extract_raw(invoices[0]["text"])
print(raw)

"""
EXPECTED OUTPUT (shape, not exact text — live model call)
---------------
A JSON string roughly matching the Invoice shape, e.g.:
{"invoice_number": "1001", "customer": "Acme Corp", "currency": "USD",
 "line_items": [{"description": "Laptop Stand", "quantity": 3, "unit_price": 25}, ...],
 "total": 86}
"""


{
  "invoice_number": "INV-1001",
   "customer": "Acme Corp.",
   "currency": "USD",
   "line_items": [
      {
         "description": "Laptop Stand",
         "quantity": 3,
         "unit_price": 25
      },
      {
         "description": "USB-C Hub",
         "quantity": 2,
         "unit_price": 18
      }
   ],
   "total": 86
}


'\nEXPECTED OUTPUT (shape, not exact text — live model call)\n---------------\nA JSON string roughly matching the Invoice shape, e.g.:\n{"invoice_number": "1001", "customer": "Acme Corp", "currency": "USD",\n "line_items": [{"description": "Laptop Stand", "quantity": 3, "unit_price": 25}, ...],\n "total": 86}\n'

In [16]:
# === Self-check (new — was missing before) ===
# Can only check shape, not content, since this is a live model call.
assert isinstance(raw, str) and len(raw) > 0, "extract_raw should return a non-empty string"
try:
    parsed_preview = json.loads(raw)
    print("extract_raw: OK -> returned valid JSON with keys", sorted(parsed_preview.keys()))
except json.JSONDecodeError:
    print("extract_raw: OK (ran without error) -> but output is NOT valid JSON yet;")
    print("that's expected sometimes — Part 4/5 below is exactly what handles this.")

"""
EXPECTED OUTPUT
---------------
extract_raw: OK -> returned valid JSON with keys [...]
(or the "not valid JSON yet" message — also a valid outcome at this stage)
"""


extract_raw: OK -> returned valid JSON with keys ['currency', 'customer', 'invoice_number', 'line_items', 'total']


'\nEXPECTED OUTPUT\n---------------\nextract_raw: OK -> returned valid JSON with keys [...]\n(or the "not valid JSON yet" message — also a valid outcome at this stage)\n'

## 4. Validate one record (no retry yet)

**Concept:** A model's JSON output can fail in two ways: it may not parse as JSON at all
(`json.JSONDecodeError`), or it may parse but not match the `Invoice` schema (`ValidationError`).
Before building a full retry pipeline, it's worth seeing what a single validation attempt looks
like on its own — success returns a usable `Invoice` object; failure returns an error message you
can inspect.

**Question:** Let's write a function that takes the raw JSON text from `extract_raw`, parses it,
and validates it against `Invoice` — returning the invoice on success, or `None` plus the error
message on failure.

**Task:** Implement `validate_invoice(raw)`:
1. In a `try` block, parse `raw` with `json.loads(raw)`.
2. Construct `Invoice(**data)`.
3. Return `(invoice, None)`.
4. In an `except (json.JSONDecodeError, ValidationError) as e` block, return `(None, str(e))`.


In [17]:
def validate_invoice(raw: str) -> tuple[Invoice | None, str | None]:
    # Parse raw as JSON, construct an Invoice, and return (invoice, None) on success
    # or (None, str(e)) on failure ===
    try:
        data = json.loads(raw)
        invoice = Invoice(**data)
        return invoice, None
    except (json.JSONDecodeError, ValidationError) as e:
        return None, str(e)

invoice, error = validate_invoice(raw)
print(invoice if invoice else error)


invoice_number='INV-1001' customer='Acme Corp.' currency='USD' line_items=[LineItem(description='Laptop Stand', quantity=3.0, unit_price=25.0), LineItem(description='USB-C Hub', quantity=2.0, unit_price=18.0)] total=86.0


In [ ]:
# === Self-check ===
assert invoice is not None or error is not None, "Expected either an invoice or an error, not neither"
print("validate_invoice: OK")

"""
EXPECTED OUTPUT
---------------
validate_invoice: OK
"""


## 5. Add a repair loop and run it on every record

**Concept:** A single validation attempt isn't enough for messy real-world data — some records
will fail on the first try. A repair loop catches the error, sends it back to the model along with
the original text, and asks for a corrected response, retrying up to `max_retries` times before
giving up on that record. A small delay between records keeps you under Groq's free-tier rate
limit.

**Question:** Let's extend this into a full pipeline: if validation fails, send the error back to
the model and ask it to fix its JSON, retrying a limited number of times, then run it across all
18 records.

**Task — in the `try` block:**
1. Set `data = json.loads(raw)`.
2. Return `Invoice(**data), errors`.

**Task — in the `except (json.JSONDecodeError, ValidationError) as e` block:**
1. Append `str(e)` to `errors`.
2. If `attempt == max_retries`, return `None, errors`.
3. Otherwise, build a string `repair_prompt` containing the error `e` and the original `text`, requesting corrected JSON with the same field names.
4. Call `completion(model="groq/llama-3.1-8b-instant", messages=[{"role": "user", "content": repair_prompt}], response_format={"type": "json_object"})`.
5. Set `raw = resp.choices[0].message.content`.
6. Outside `parse_invoice`, loop over `invoices`, call `parse_invoice(record["text"])` for each, append `{"id": record["id"], "invoice": invoice, "errors": errs}` to `results`, and `time.sleep(2)` between records.


In [18]:
def parse_invoice(text: str, max_retries: int = 2) -> tuple[Invoice | None, list[str]]:
    """Returns (parsed Invoice or None, list of attempt errors)."""
    errors = []
    raw = extract_raw(text)
    for attempt in range(max_retries + 1):
        try:
            # json.loads the raw string, validate/construct an Invoice, return (invoice, errors)
            data = json.loads(raw)
            invoice = Invoice(**data)
            return invoice, errors

        except (json.JSONDecodeError, ValidationError) as e:
            # Record the error; return (None, errors) on last attempt; otherwise repair and retry
            errors.append(str(e))
            if attempt == max_retries:
                return None, errors
            repair_prompt = f"""The following JSON failed validation with error: {e}

Original text:
{text}

Previous JSON attempt:
{raw}

Return corrected JSON with EXACTLY these fields: invoice_number, customer, currency,
line_items (list of objects with description, quantity, unit_price), total.
Return ONLY valid JSON, no markdown fences, no commentary."""
            resp = completion(
                model="groq/llama-3.1-8b-instant",
                messages=[{"role": "user", "content": repair_prompt}],
                response_format={"type": "json_object"},
            )
            raw = resp.choices[0].message.content
    return None, errors

results = []
for record in invoices:
    invoice, errs = parse_invoice(record["text"])
    results.append({"id": record["id"], "invoice": invoice, "errors": errs})
    time.sleep(2)  # stay under Groq's free-tier tokens-per-minute limit

success = sum(1 for r in results if r["invoice"] is not None)
print(f"Parsed {success}/{len(results)} records successfully.")

"""
EXPECTED OUTPUT (real numbers depend on the live model's behaviour this run)
---------------
Parsed <N>/18 records successfully.
Records #10 (deliberately wrong math), #11/#15 (mixed currencies), #13/#16 (missing totals)
are the ones most likely to still fail after retries — worth inspecting their `errors` list.
"""


Parsed 18/18 records successfully.


"\nEXPECTED OUTPUT (real numbers depend on the live model's behaviour this run)\n---------------\nParsed <N>/18 records successfully.\nRecords #10 (deliberately wrong math), #11/#15 (mixed currencies), #13/#16 (missing totals)\nare the ones most likely to still fail after retries — worth inspecting their `errors` list.\n"

In [19]:
# Self-check
assert len(results) == len(invoices), "Should have one result per input record"
assert success >= 1, "At least one record should parse successfully"
for r in results:
    if r["invoice"] is not None:
        assert isinstance(r["invoice"], Invoice)
print(f"parse_invoice: OK -> {success}/{len(results)} parsed, {len(results) - success} need the repair path reviewed.")

"""
EXPECTED OUTPUT
---------------
parse_invoice: OK -> <N>/18 parsed, <18-N> need the repair path reviewed.
"""


parse_invoice: OK -> 18/18 parsed, 0 need the repair path reviewed.


'\nEXPECTED OUTPUT\n---------------\nparse_invoice: OK -> <N>/18 parsed, <18-N> need the repair path reviewed.\n'

## 6. Reflection / stretch goal (optional)

**Pitfall (from the session notes):** validation errors from free-tier models can be noisy.
**Stretch goal:** budget extra repair attempts — bump `max_retries` and log which record IDs still
fail after all retries. Inspect their `errors` list: what pattern do the persistent failures share
(e.g. two currencies in one line, no total at all)? Adjust your prompt or schema to handle that
pattern.


## Capstone milestone tie-in

This lab satisfies **Milestone 1 — "Provider-agnostic LLM client + structured intake"**, the validated `Invoice` objects here are the structured intake layer your project's data pipeline will build on.
